In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.external_data_encyclopedia import ExternalDataEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

ed = ExternalDataEncyclopedia(
    external_data_path=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data")
)

In [2]:
refine_plus_export_pool = Path("refine_plus_export_pool")
combined_db_export_pool = Path("combined_db_export_pool")
agg_export_pool = Path("agg_export_pool/pass_00")
pass_00_path: Path = combined_db_export_pool / "pass_00"
pass_01_path: Path = combined_db_export_pool / "pass_01"
pipeline_from_here_needs_running : bool = False

refine_plus_export_pool.mkdir(exist_ok=True, parents=True)
agg_export_pool.mkdir(exist_ok=True, parents=True)
pass_00_path.mkdir(exist_ok=True, parents=True)
pass_01_path.mkdir(exist_ok=True, parents=True)

## Fill export pool

In [3]:
from boulder_statistics.refinement_plus.bulk_parse_data_tir_maps import DataTirMaps
from boulder_statistics.refinement_plus.project_facets import ProjectFacets
from boulder_statistics.refinement_plus.bulk_parse_data_tir_maps import (
    TIR_MEASUREMENT_NAMES,
    TIR_SIGMA_MEASUREMENT_NAMES,
)

data_tir_maps: DataFrame = DataTirMaps.bulk_parse(ed)

for mission_phase in ["detailed_survey", "recona", "reconb"]:

    pf = ProjectFacets(
        mesh_folder=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data\bennu_models"),
        export_pool_folder=refine_plus_export_pool,
        measurement_types_of_interest = TIR_MEASUREMENT_NAMES + TIR_SIGMA_MEASUREMENT_NAMES,
        facet_maps=data_tir_maps,
        mission_phase=mission_phase,
        instrument_type='TIR',
        dp=dp
    )

    if (not pf.result_export_folder.exists()) or pipeline_from_here_needs_running:
        pipeline_from_here_needs_running = True
        res: None | Path = pf.process_mission_phase()

c:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
Processing chunks: 100%|██████████| 24/24 [00:36<00:00,  1.53s/it]


In [4]:
from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import DataVnirMaps
from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import (
    VNIR_MEASUREMENT_NAMES,
    VNIR_SIGMA_MEASUREMENT_NAMES,
)

data_vnir_maps: DataFrame = DataVnirMaps.bulk_parse(ed)

for mission_phase in ["detailed_survey", "recona", "reconc"]:

    pf = ProjectFacets(
        mesh_folder=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data\bennu_models"),
        export_pool_folder=refine_plus_export_pool,
        measurement_types_of_interest = VNIR_MEASUREMENT_NAMES + VNIR_SIGMA_MEASUREMENT_NAMES,
        facet_maps=data_vnir_maps,
        mission_phase=mission_phase,
        instrument_type='VNIR',
        dp=dp
    )

    if (not pf.result_export_folder.exists()) or pipeline_from_here_needs_running:
        pipeline_from_here_needs_running = True
        res: None | Path = pf.process_mission_phase()

Processing chunks: 100%|██████████| 24/24 [00:48<00:00,  2.03s/it]


In [5]:
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools
from boulder_statistics.refinement_plus.refinement_tools import restrict_by_clipped_sigma_log_space

def handle_mesh(chunk : QCubeChunk) -> np.ndarray:
    areas: np.ndarray = ChunkingTools.extract_chunks(dp.Phi_mesh, chunk, ["area"])[0]
    areas *= 1e6 # Convert to m^2
    areas = restrict_by_clipped_sigma_log_space(areas)

    return areas

area_export_folder: Path = refine_plus_export_pool / "00_add_areas"

if (not area_export_folder.exists()) or pipeline_from_here_needs_running:
    pipeline_from_here_needs_running = True
    
    ChunkingTools.append_by_chunks(
        dp.combined_atlas.select("i", "j", "face"),
        refine_plus_export_pool / "00_add_areas",
        "area",
        handle_mesh,
        chunks = QCubeChunk.generate(depth=2)
    )

Processing chunks: 100%|██████████| 96/96 [01:10<00:00,  1.36it/s]


## Combine pass 00

In [ ]:
import os
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools

lfs_to_join: list[LazyFrame] = [
    pl.scan_parquet(refine_plus_export_pool / table_name)
    for table_name in os.listdir(refine_plus_export_pool)
] + [
    dp.combined_atlas,
    dp.combined_mask.unique(
    subset=["i", "j", "face"], keep="first" # Sometimes duplicates can occur but are rare
    ).rename({
        "lod_level": "detection_lod_level",
        "lod_code": "detection_lod_code",
        "row_id" : "boulder_id"
    })
]

for lf in lfs_to_join:
    print(lf.collect_schema().names())

if not pass_00_path.exists() or pipeline_from_here_needs_running:
    pipeline_from_here_needs_running = True
    ChunkingTools.join_in_chunks(
        export_folder = pass_00_path,
        lfs_to_join = lfs_to_join,
        chunks = QCubeChunk.generate(depth=4),
    )

['i', 'j', 'face', 'area']
['i', 'j', 'face', 'TIR detailed_survey tri_num', 'TIR detailed_survey band depth 350', 'TIR detailed_survey band depth 440', 'TIR detailed_survey slope 1000', 'TIR detailed_survey ratio 1000', 'TIR detailed_survey sigma band depth 350', 'TIR detailed_survey sigma band depth 440', 'TIR detailed_survey sigma slope 1000', 'TIR detailed_survey sigma ratio 1000']
['i', 'j', 'face', 'TIR recona tri_num', 'TIR recona band depth 350', 'TIR recona band depth 440', 'TIR recona slope 1000', 'TIR recona ratio 1000', 'TIR recona sigma band depth 350', 'TIR recona sigma band depth 440', 'TIR recona sigma slope 1000', 'TIR recona sigma ratio 1000']
['i', 'j', 'face', 'TIR reconb tri_num', 'TIR reconb band depth 350', 'TIR reconb band depth 440', 'TIR reconb slope 1000', 'TIR reconb ratio 1000', 'TIR reconb sigma band depth 350', 'TIR reconb sigma band depth 440', 'TIR reconb sigma slope 1000', 'TIR reconb sigma ratio 1000']
['i', 'j', 'face', 'VNIR detailed_survey tri_num'

Joining:  63%|██████▎   | 973/1536 [05:40<02:34,  3.64it/s]

## Agg pass 00

In [ ]:
from pathlib import Path


agg_exprs = [
    ((pl.col("32bit_reflectance").max() - pl.col("32bit_reflectance").mean())
     / pl.col("32bit_reflectance").std()).alias("Gamma"),
     (pl.col("32bit_reflectance").max() / pl.col("32bit_reflectance").mean() - 1).alias("Tau"),

    pl.col("area").sum().alias("Area"),

    pl.col("positions_x").mean().alias("center_x"),
    pl.col("positions_x").min().alias("min_x"),
    pl.col("positions_x").max().alias("max_x"),

    pl.col("positions_y").mean().alias("center_y"),
    pl.col("positions_y").min().alias("min_y"),
    pl.col("positions_y").max().alias("max_y"),

    pl.col("positions_z").mean().alias("center_z"),
    pl.col("positions_z").min().alias("min_z"),
    pl.col("positions_z").max().alias("max_z"),

    pl.len().alias("number_of_samples"),
]

agg_pass_00_boulder_id_path: Path = agg_export_pool / "boulder_id.parquet"
if (not agg_pass_00_boulder_id_path.exists()) or pipeline_from_here_needs_running:
    pipeline_from_here_needs_running = True
    agg_df = pl.scan_parquet(pass_00_path).group_by(
        "boulder_id").agg(agg_exprs).collect(engine="streaming")

    agg_df.write_parquet(agg_pass_00_boulder_id_path)

In [ ]:
filter_df: DataFrame = pl.scan_parquet(pass_00_path).filter(pl.col("boulder_id") < 40).collect()
cols_to_group: list[str] = [col_name for col_name in filter_df.columns if "tri_num" in col_name]

for col in cols_to_group:
    agg_pass_00_col_path: Path = agg_export_pool / f"{col}.parquet"

    if agg_pass_00_col_path.exists() and (pipeline_from_here_needs_running == False):
        pipeline_from_here_needs_running = True
        continue

    print(f"Joining for col {col}")
    df: DataFrame = pl.scan_parquet(pass_00_path).select(col).group_by(
        col
    ).agg(pl.len().alias(f"{col} alpha")).collect(engine="streaming")
    df.write_parquet(agg_pass_00_col_path)

## Combine pass 01

In [ ]:
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools
import os 

if (not pass_01_path.exists()) or pipeline_from_here_needs_running:
    pipeline_from_here_needs_running = True
    ChunkingTools.join_full_with_aggs(
        export_folder = pass_01_path,
        full_db=pl.scan_parquet(pass_00_path),
        aggs_to_join_with={
            (Path(agg_name).stem,) : pl.read_parquet(agg_export_pool / agg_name)
            for agg_name in os.listdir(agg_export_pool)
        },
        chunks = QCubeChunk.generate(depth=2)
    )